In [1]:
!pip install torch torchvision torchaudio


In [2]:
!pip install transformers beautifulsoup4 pandas requests numpy

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import requests
from bs4 import BeautifulSoup
import re

In [5]:
tokenizer = AutoTokenizer.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

model = AutoModelForSequenceClassification.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

In [6]:
tokens = tokenizer.encode('It was good but couldve been much better.', return_tensors='pt')
result = model(tokens)
result.logits
int(torch.argmax(result.logits))+1

3

In [2]:
import os
import pandas as pd
from bs4 import BeautifulSoup
from nltk.tokenize import sent_tokenize
import re
import csv

# Define paths
input_csv_temp = "hyundai-ratings"
input_csv_path = "/home/madhavbpanicker/Documents/Scrape_project/Website-Data-Raw/scraped_results-hyundai-ratings.csv"  # Replace with the actual path to your raw csv
 
output_dir = "/home/madhavbpanicker/Documents/Scrape_project/Website-Data-Trimmed"
output_filename = f"Website-Data-Trimmed-{input_csv_temp}"
output_csv_path = os.path.join(output_dir, output_filename)

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Load the raw csv
data = pd.read_csv(input_csv_path)

# Function to clean HTML content
def clean_html(raw_html):
    if pd.isna(raw_html):  # Skip if the content is NaN
        return ""
    soup = BeautifulSoup(raw_html, "html.parser")
    # Remove script and style elements
    for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()
    # Get clean text
    text = soup.get_text()
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Function to process each row and split content into sentences
def process_row(row):
    cleaned_text = clean_html(row["page_content"])
    if not cleaned_text:  # Skip if no content
        return []
    sentences = sent_tokenize(cleaned_text)
    # Duplicate the other row data for each sentence
    rows = [
        {
            "Title": row["Title"],
            "Link": row["Link"],
            "Description": row["Description"],
            "Date": row["Date"],
            "Sentence": sentence,
        }
        for sentence in sentences
    ]
    return rows

# Process all rows and expand into individual sentences
processed_data = []
for _, row in data.iterrows():
    processed_data.extend(process_row(row))

# Convert processed data into a DataFrame
processed_df = pd.DataFrame(processed_data)

# Save the processed data to the new csv
processed_df.to_csv(output_csv_path, index=False)

print(f"Data has been cleaned and saved to {output_csv_path}")

Data has been cleaned and saved to /home/madhavbpanicker/Documents/Scrape_project/Website-Data-Trimmed/hyundai-ratings


In [ ]:
# Define search terms
search_terms = ["hyundai ratings", "hyundai", "ratings", "review of hyundai"]

# Process all rows and expand into individual sentences
processed_data = []
for _, row in data.iterrows():
    processed_data.extend(process_row(row, search_terms))

# Convert processed data into a DataFrame
processed_df = pd.DataFrame(processed_data)

# Save the processed data to the new csv
processed_df.to_csv(output_csv_path, index=False)

print(f"Filtered and cleaned data has been saved to {output_csv_path}")

In [9]:
import os
import json
from bs4 import BeautifulSoup
from textblob import TextBlob
import re
import pandas as pd

# Define paths
input_json_temp = "toyota-\"forum\"-reviews"
project_name = "reputation-management"
input_json_path = f"/home/madhavbpanicker/Documents/Scrape_project/Website-Data-Raw/{project_name}-scraped_results-{input_json_temp}-.json"  # Replace with the actual path to your raw JSON

output_dir = "/home/madhavbpanicker/Documents/Scrape_project/Website-Data-Trimmed"
output_filename = f"{project_name}-Website-Data-Trimmed-{input_json_temp}.json"
output_json_path = os.path.join(output_dir, output_filename)

# Ensure the output directory exists
os.makedirs(output_dir, exist_ok=True)

# Load the raw JSON file
with open(input_json_path, 'r') as file:
    data = json.load(file)

# Function to clean HTML content
def clean_html(raw_html):
    if not raw_html:  # Skip if the content is None or empty
        return ""
    soup = BeautifulSoup(raw_html, "html.parser")
    # Remove script and style elements
    for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()
    # Get clean text
    text = soup.get_text()
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Function to process each segment (e.g., comment or section)
def process_segment(segment):
    cleaned_text = clean_html(segment.get("content", ""))  # Assuming "content" holds the relevant HTML or text
    if not cleaned_text:  # Skip if no content
        return None

    # Perform sentiment analysis
    sentiment = TextBlob(cleaned_text).sentiment

    # Return the processed data
    return {
        "Title": segment.get("Title", ""),
        "Link": segment.get("Link", ""),
        "Description": segment.get("Description", ""),
        "Date": segment.get("Date", ""),
        "CleanedContent": cleaned_text,
        "Sentiment": {
            "Polarity": sentiment.polarity,
            "Subjectivity": sentiment.subjectivity
        }
    }

# Process all segments
processed_data = []
for segment in data:
    processed_segment = process_segment(segment)
    if processed_segment:
        processed_data.append(processed_segment)

# Save the processed data to the new JSON file
# If `processed_data` is a DataFrame, convert it to a list of dictionaries first
if isinstance(processed_data, pd.DataFrame):
    processed_data = processed_data.to_dict(orient="records")

# Save the processed data to the new JSON file
with open(output_json_path, 'w') as file:
    json.dump(processed_data, file, indent=4)
print(f"Data has been cleaned, analyzed, and saved to {output_json_path}")


Data has been cleaned, analyzed, and saved to /home/madhavbpanicker/Documents/Scrape_project/Website-Data-Trimmed/reputation-management-Website-Data-Trimmed-toyota-"forum"-reviews.json
